# Pipeline node 02 — Train YOLOv8n on the VLM-annotated set

Production-parallel of notebook [`../04-train-yolo.ipynb`](../04-train-yolo.ipynb). Pulls the YOLO-format dataset produced by node 01 from S3, fine-tunes `yolov8n.pt`, uploads `best.pt` back to S3.

**S3 contract**
- *Input:*  `s3://$BUCKET/$DATASET_PREFIX/{images,labels}/train/*`, `dataset.yaml`
- *Output:* `s3://$BUCKET/$MODEL_PREFIX/$MODEL_VERSION/best.pt`

**Env vars expected**
- `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_S3_ENDPOINT`, `AWS_DEFAULT_REGION`, `AWS_S3_BUCKET`
- `DATASET_PREFIX` (default `workshop-dataset`), `MODEL_PREFIX` (default `models/ppe-yolov8n-vlm`), `MODEL_VERSION` (default `v1`)
- `EPOCHS` (default `20`), `IMG_SIZE` (default `640`), `BATCH` (default `16`)

In [ ]:
%%capture
!pip install ultralytics boto3==1.35.55 botocore==1.35.55

In [ ]:
import os
import shutil
from pathlib import Path

import boto3
import botocore
import torch

BUCKET         = os.environ["AWS_S3_BUCKET"]
DATASET_PREFIX = os.environ.get("DATASET_PREFIX", "workshop-dataset").rstrip("/")
MODEL_PREFIX   = os.environ.get("MODEL_PREFIX",   "models/ppe-yolov8n-vlm").rstrip("/")
MODEL_VERSION  = os.environ.get("MODEL_VERSION",  "v1")
EPOCHS         = int(os.environ.get("EPOCHS",   "100"))
IMG_SIZE       = int(os.environ.get("IMG_SIZE", "640"))
BATCH          = int(os.environ.get("BATCH",    "16"))
LR0            = float(os.environ.get("LR0",       "0.001"))
FREEZE         = int(os.environ.get("FREEZE",     "10"))
PATIENCE       = int(os.environ.get("PATIENCE",   "20"))

STAGE = Path("/tmp/ppe-train")
DATA_DIR = STAGE / "data"
RUNS_DIR = STAGE / "runs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

session = boto3.session.Session(
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)
s3 = session.resource(
    "s3",
    config=botocore.client.Config(signature_version="s3v4"),
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
)
bucket = s3.Bucket(BUCKET)

print(
    f"CUDA available: {torch.cuda.is_available()}; "
    f"epochs={EPOCHS}, imgsz={IMG_SIZE}, batch={BATCH}, "
    f"lr0={LR0}, freeze={FREEZE}, patience={PATIENCE}"
)

## Download the YOLO-format dataset from S3

In [ ]:
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True)

count = 0
for obj in bucket.objects.filter(Prefix=f"{DATASET_PREFIX}/"):
    rel = obj.key[len(DATASET_PREFIX) + 1 :]
    if not rel:
        continue
    dst = DATA_DIR / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    bucket.download_file(obj.key, str(dst))
    count += 1

print(f"Downloaded {count} objects -> {DATA_DIR}")
print("--- dataset.yaml ---")
print((DATA_DIR / "dataset.yaml").read_text())

## Fine-tune YOLOv8n

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=str(DATA_DIR / "dataset.yaml"),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    lr0=LR0,
    freeze=FREEZE,
    patience=PATIENCE,
    project=str(RUNS_DIR),
    name="ppe-vlm",
    exist_ok=True,
)

SAVE_DIR = Path(model.trainer.save_dir)
best_pt = SAVE_DIR / "weights" / "best.pt"
assert best_pt.is_file(), f"Training did not produce {best_pt}"
print("Best weights:", best_pt, f"({best_pt.stat().st_size / 1e6:.1f} MB)")
print("Run artifacts:", SAVE_DIR)

## Upload `best.pt` to S3

Node 03 reads this S3 URI from the `MODEL_URI` env var.

In [ ]:
s3_prefix = f"{MODEL_PREFIX}/{MODEL_VERSION}"
bucket.upload_file(str(best_pt), f"{s3_prefix}/best.pt")

# Package a data.yaml alongside so the serving runtime knows class names.
bucket.upload_file(str(DATA_DIR / "dataset.yaml"), f"{s3_prefix}/data.yaml")

# Upload training provenance so node 03 can enrich the Model Registry entry.
for extra in ("args.yaml", "results.csv"):
    src = SAVE_DIR / extra
    if src.is_file():
        bucket.upload_file(str(src), f"{s3_prefix}/{extra}")
        print(f"Uploaded {extra} ({src.stat().st_size} bytes)")
    else:
        print(f"(skipped missing {extra})")

model_uri = f"s3://{BUCKET}/{s3_prefix}"
print("Model URI:", model_uri)

# Emit for the next pipeline node.
Path("model_uri.txt").write_text(model_uri)